# Two-Tower Retrieval Model

---

## What this notebook does (plain English)

We are building a **product search system** — like Amazon's search bar.

When a user types **"red dress for a wedding"**, the system should return the most relevant products.

We do this with a **Two-Tower model**:
- **Query Tower** — reads a search query and converts it into a small list of numbers (a vector)
- **Product Tower** — reads a product title and converts it into a small list of numbers (a vector)
- We train both towers so that **relevant query-product pairs end up with similar vectors**
- At search time: encode a query → find the nearest product vectors → return those products

```
"red dress"  →  Query Tower  →  [0.2, -0.5, 0.8, ...]  ←── these should be close
"Red Evening Gown"  →  Product Tower  →  [0.3, -0.4, 0.7, ...]

"red dress"  →  Query Tower  →  [0.2, -0.5, 0.8, ...]  ←── these should be far apart
"Car Engine Oil"  →  Product Tower  →  [-0.9, 0.1, -0.2, ...]
```


## PART 1 — Install & Import Libraries


In [ ]:
import json


import numpy as np
import pandas as pd
import gc
import warnings
import os

import tensorflow as tf
from tensorflow import keras

from keras.layers import Input
from keras.layers import Embedding
from keras.layers import GlobalAveragePooling1D
from keras.layers import Dense
from keras.layers import Dot
from keras.layers import Lambda

from keras.models import Model
from keras.optimizers import Adam
from keras.callbacks import ModelCheckpoint

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

!pip install faiss-cpu
import faiss

from IPython.display import display

warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', None)

print("TensorFlow version:", tf.__version__)
print("All libraries loaded successfully!")


## PART 2 — Load the Data


In [ ]:

df_products = pd.read_parquet(
    '/kaggle/input/amazon-query-product-search/shopping_queries_dataset_products.parquet'
)
df_products = df_products[['product_id', 'product_title']].drop_duplicates()

df_labels = pd.read_csv(
    '/kaggle/input/amazon-query-product-search/dataset_150k.csv'
)

df_dataset = pd.merge(df_labels, df_products, on='product_id')

print("Dataset shape:", df_dataset.shape)
print("\nColumns:", df_dataset.columns.tolist())
print("\nLabel distribution:")
print(df_dataset['esci_label'].value_counts())

df_dataset.head()


## PART 3 — Convert Labels & Split Train / Validation


In [ ]:
df_dataset['label'] = (df_dataset['esci_label'] == 'E').astype('float32')

train_df = df_dataset[df_dataset['split'] != 'test'].reset_index(drop=True)
val_df   = df_dataset[df_dataset['split'] == 'test'].reset_index(drop=True)

print(f"Training rows   : {len(train_df):,}")
print(f"Validation rows : {len(val_df):,}")
print(f"\nPositive rate (train): {train_df['label'].mean():.2%}")
print(f"Positive rate (val)  : {val_df['label'].mean():.2%}")

train_labels = train_df['label'].values
val_labels   = val_df['label'].values


## PART 4 — Tokenize Raw Text


In [ ]:
VOCAB_SIZE      = 50000
MAX_QUERY_LEN   = 10
MAX_PRODUCT_LEN = 30


all_queries  = df_dataset['query'].tolist()
all_products = df_dataset['product_title'].tolist()

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<UNK>')

tokenizer.fit_on_texts(all_queries + all_products)

print(f"Vocabulary size: {len(tokenizer.word_index):,} unique words found")
print(f"We keep the top {VOCAB_SIZE:,} most common ones")


def tokenize_and_pad(texts, max_length):
    word_id_sequences = tokenizer.texts_to_sequences(texts)
    padded_sequences  = pad_sequences(
        word_id_sequences,
        maxlen=max_length,
        padding='post',
        truncating='post'
    )
    return padded_sequences


train_queries  = tokenize_and_pad(train_df['query'].tolist(),         MAX_QUERY_LEN)
train_products = tokenize_and_pad(train_df['product_title'].tolist(), MAX_PRODUCT_LEN)

val_queries    = tokenize_and_pad(val_df['query'].tolist(),           MAX_QUERY_LEN)
val_products   = tokenize_and_pad(val_df['product_title'].tolist(),   MAX_PRODUCT_LEN)


print(f"\nTrain query array shape   : {train_queries.shape}")
print(f"Train product array shape : {train_products.shape}")
print(f"\nExample -- 'red dress for a wedding' tokenized:")
print(tokenize_and_pad(['red dress for a wedding'], MAX_QUERY_LEN)[0])


train_inputs = [train_queries, train_products]
val_inputs   = [val_queries,   val_products]


## PART 5 — Build the Two-Tower Model


In [ ]:
EMBED_DIM  = 64
OUTPUT_DIM = 16


def build_tower(input_length, name_prefix):

    inp = Input(shape=(input_length,), name=f'input_{name_prefix}')

    word_vectors = Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBED_DIM,
        name=f'embed_{name_prefix}'
    )(inp)

    sentence_vector = GlobalAveragePooling1D(name=f'pool_{name_prefix}')(word_vectors)

    compressed_vector = Dense(OUTPUT_DIM, activation='linear', name=f'dense_{name_prefix}')(sentence_vector)

    normalized_vector = Lambda(
        lambda v: tf.keras.backend.l2_normalize(v, axis=-1),
        name=f'normalize_{name_prefix}'
    )(compressed_vector)

    return inp, normalized_vector


input_query, output_query = build_tower(MAX_QUERY_LEN, 'query')

input_product, output_product = build_tower(MAX_PRODUCT_LEN, 'product')


similarity_score = Dot(axes=1, normalize=True, name='cosine_similarity')(
    [output_query, output_product]
)


match_score = Lambda(lambda score: (score + 1.0) / 2.0, name='match_score')(similarity_score)

model = Model(
    inputs=[input_query, input_product],
    outputs=match_score,
    name='two_tower_model'
)

model.summary()


## PART 6 — Train the Model


In [ ]:
CHECKPOINT_PATH = 'best_two_tower.weights.h5'

checkpoint = ModelCheckpoint(
    CHECKPOINT_PATH,
    monitor='val_loss',
    verbose=1,
    save_best_only=True,
    save_weights_only=True,
    mode='min'
)


model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)


history = model.fit(
    train_inputs,
    train_labels,
    epochs=10,
    batch_size=128,
    validation_data=(val_inputs, val_labels),
    callbacks=[checkpoint],
    class_weight={0: 1, 1: 3}
)

print("\nTraining complete!")


In [ ]:
model.load_weights(CHECKPOINT_PATH)
print("Best model weights loaded.")


## PART 7 — Plot Training History


In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['loss'],     label='Training Loss',   linewidth=2)
ax1.plot(history.history['val_loss'], label='Validation Loss', linewidth=2, linestyle='--')
ax1.set_title('Loss over Epochs', fontsize=14)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['accuracy'],     label='Training Accuracy',   linewidth=2)
ax2.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2, linestyle='--')
ax2.set_title('Accuracy over Epochs', fontsize=14)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## PART 8 — Extract Product Embeddings


In [ ]:
df_all_products = df_dataset[['product_id', 'product_title']].drop_duplicates().reset_index(drop=True)

df_all_products['pid'] = range(len(df_all_products))

print(f"Unique products to embed: {len(df_all_products):,}")


product_tower_model = Model(
    inputs=input_product,
    outputs=model.get_layer('normalize_product').output,
    name='product_tower'
)


all_product_sequences = tokenize_and_pad(
    df_all_products['product_title'].tolist(),
    MAX_PRODUCT_LEN
)

print(f"Product sequences shape: {all_product_sequences.shape}")


product_embeddings = product_tower_model.predict(all_product_sequences, batch_size=512, verbose=1)

print(f"\nProduct embeddings shape: {product_embeddings.shape}")
print("Each product is now represented as a 16-dimensional vector")


In [ ]:
embedding_column_names = [f'dim_{i}' for i in range(OUTPUT_DIM)]

df_product_embeddings = pd.DataFrame(product_embeddings, columns=embedding_column_names)

df_product_embeddings['product_id']    = df_all_products['product_id'].values
df_product_embeddings['product_title'] = df_all_products['product_title'].values
df_product_embeddings['pid']           = df_all_products['pid'].values

print("Product embedding table:")
df_product_embeddings.head()


## PART 9 — Build the Product Search Index (FAISS)


In [ ]:
product_search_index = faiss.IndexFlatIP(OUTPUT_DIM)

index_to_pid = df_product_embeddings['pid'].tolist()

index_to_pid_as_ints = []
for x in index_to_pid:
    index_to_pid_as_ints.append(int(x))

with open('index_to_pid.json', 'w') as f:
    json.dump(index_to_pid_as_ints, f)
print("Saved: index_to_pid.json")

pid_to_title      = {}
pid_to_product_id = {}

for _, row in df_product_embeddings.iterrows():
    pid = int(row['pid'])
    pid_to_title[pid]      = row['product_title']
    pid_to_product_id[pid] = row['product_id']

vectors = df_product_embeddings[embedding_column_names].values.astype('float32')

faiss.normalize_L2(vectors)

product_search_index.add(vectors)
print(f"Added {product_search_index.ntotal:,} product vectors to FAISS index")

faiss.write_index(product_search_index, 'product_search_index.faiss')
print("Saved: product_search_index.faiss")

product_search_index = faiss.read_index('product_search_index.faiss')
print("FAISS product search index ready!")


## PART 10 — Quality Check: Are Similar Products Close Together?


In [ ]:

TOP_K         = 5
ROWS_TO_CHECK = 10

results = []

for i in range(ROWS_TO_CHECK):
    row = df_product_embeddings.iloc[i]
    pid           = int(row.pid)
    product_title = row.product_title

    raw_vec = df_product_embeddings[embedding_column_names].iloc[i].values
    vec = raw_vec.astype('float32').reshape(1, -1)

    faiss.normalize_L2(vec)

    _, neighbor_index_array = product_search_index.search(vec, TOP_K + 1)
    neighbor_indices = neighbor_index_array[0]

    nearest_titles = []
    for faiss_row_idx in neighbor_indices:
        if faiss_row_idx == -1:
            continue

        real_pid = int(index_to_pid[faiss_row_idx])

        if real_pid == pid:
            continue

        nearest_titles.append(pid_to_title.get(real_pid, 'UNKNOWN'))

        if len(nearest_titles) == TOP_K:
            break

    results.append([product_title] + nearest_titles)

columns = ['product']
for i in range(TOP_K):
    columns.append(f'similar_{i+1}')

df_quality_check = pd.DataFrame(results, columns=columns)

print("Product similarity quality check:")
print("(Each row shows a product and its 5 most similar products)")
display(df_quality_check)


## PART 11 — Create a Query Tower Sub-Model


In [ ]:
query_tower_model = Model(
    inputs=input_query,
    outputs=model.get_layer('normalize_query').output,
    name='query_tower'
)

print("Query Tower sub-model ready!")
query_tower_model.summary()


## PART 12 — Search Function


In [ ]:
def search(query_text, top_k=10):

    word_id_list  = tokenizer.texts_to_sequences([query_text])
    padded_query  = pad_sequences(
        word_id_list,
        maxlen=MAX_QUERY_LEN,
        padding='post',
        truncating='post'
    )

    query_embedding = query_tower_model.predict(padded_query, verbose=0)[0]

    query_vec_f32 = query_embedding.reshape(1, -1).astype('float32')

    faiss.normalize_L2(query_vec_f32)

    distances_array, row_indices_array = product_search_index.search(query_vec_f32, top_k)

    distances  = distances_array[0]
    faiss_rows = row_indices_array[0]

    results = []
    rank = 1

    total_results = len(faiss_rows)

    for i in range(total_results):

        faiss_row  = faiss_rows[i]
        similarity = distances[i]

        if faiss_row == -1:
            continue

        real_pid = int(index_to_pid[faiss_row])

        title = pid_to_title.get(real_pid, 'UNKNOWN')

        similarity_rounded = round(float(similarity), 4)

        one_result = {
            'rank'            : rank,
            'product_title'   : title,
            'similarity_score': similarity_rounded
        }

        results.append(one_result)

        rank = rank + 1

    return pd.DataFrame(results)


print("Search function is ready!")
print("Usage: search('your query here', top_k=10)")


## PART 13 — Run Searches!


In [ ]:
test_queries = [
    "red dress for a wedding",
    "best laptop under 1000",
    "nike running shoes for women",
    "bluetooth earbuds under 50",
    "organic skincare products",
    "gaming laptop with rtx graphics",
    "winter jacket for men",
    "coffee maker"
]

TOP_K = 5

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: '{query}'")
    print(f"{'='*60}")
    results_df = search(query, top_k=TOP_K)
    display(results_df)


## PART 14 — Save Everything for Later Use


In [ ]:
import json

tokenizer_as_json = tokenizer.to_json()
with open('tokenizer_config.json', 'w') as f:
    f.write(tokenizer_as_json)
print("Saved: tokenizer_config.json")


query_tower_model.save('query_tower.h5')
print("Saved: query_tower.h5")


product_tower_model.save('product_tower.h5')
print("Saved: product_tower.h5")


df_all_products.to_csv('product_metadata.csv', index=False)
print("Saved: product_metadata.csv")


print("\nAll files saved! To use this system later:")
print("1. Load tokenizer from tokenizer_config.json")
print("2. Load query_tower.h5")
print("3. Load product_search_index.faiss")
print("4. Load product_metadata.csv for title lookups")
print("5. Call search(query_text) to search!")


## Summary — Full System Flow

---

Here's the complete picture:

```
TRAINING TIME:

Raw query text  ──►  Tokenizer  ──►  [45, 231, 0, ...]  ──►  Query Tower  ──►  [0.2, -0.5, ...]  (16D)
                                                                                                        \
                                                                                             Cosine Sim  ──►  Match score (0 to 1)
                                                                                                        /
Raw product text ──►  Tokenizer  ──►  [12, 88, 44, ...]  ──►  Product Tower ──►  [0.3, -0.4, ...]  (16D)

Match score = (cosine similarity + 1) / 2
Loss compares this score with the true label (1=relevant, 0=not relevant)
Optimizer adjusts weights to reduce loss


SEARCH TIME:

New query text ──►  Tokenizer  ──►  Query Tower  ──►  16D vector
                                                            |
                                                    FAISS Index Search
                                                            |
                                               Top-K nearest product vectors
                                                            |
                                                   Return product titles ✅
```

| Part | What it does |
|------|--------------|
| 1 | Import libraries |
| 2 | Load and merge data |
| 3 | Convert labels to binary, split train/val |
| 4 | **Tokenize raw text** (word → integer ID, pad to fixed length) |
| 5 | **Build Two-Tower model** (Embedding → Pool → Dense → Normalize → Cosine Similarity → Match Score) |
| 6 | Train the model |
| 7 | Plot training history |
| 8 | Extract 16-dim embeddings for all products |
| 9 | Build FAISS search index from product embeddings |
| 10 | Quality check: are similar products near each other? |
| 11 | Create Query Tower sub-model |
| 12 | `search()` function |
| 13 | Run test searches |
| 14 | Save everything |
